In [43]:
#Tarea 4-5 Análisis de MLlib PySpark

'''
Falta hacer ajustes
'''

In [44]:
'''
Realizar análisis con MLlib de pyspark a tu conjunto de datos.
'''

'\nRealizar análisis con MLlib de pyspark a tu conjunto de datos.\n'

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.appName("MLlib_Analysis").getOrCreate()

# Cargar dataset
df = spark.read.csv("/content/dataset.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)


root
 |-- _c0: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: double (nullable = true)
 |-- track_genre: string (nullable = true)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+---

In [4]:
# Selección de variables
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer

# Convertir columnas numéricas que están como string
numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "key", "loudness", "mode", "speechiness",
    "acousticness", "liveness", "valence"
]

for c in numeric_cols:
    df = df.withColumn(c, col(c).cast(DoubleType()))


In [5]:
# se seleccionan a continuacion soloa las variables nuemricas
selected_cols = numeric_cols + [
    "instrumentalness", "tempo", "time_signature", "track_genre"
]

df_model = df.select(selected_cols).dropna()


In [13]:
df.select("album_name", "duration_ms").show(20, truncate=False)


+------------------------------------------------------+-----------+
|album_name                                            |duration_ms|
+------------------------------------------------------+-----------+
|Comedy                                                |230666.0   |
|Ghost (Acoustic)                                      |149610.0   |
|To Begin Again                                        |210826.0   |
|Crazy Rich Asians (Original Motion Picture Soundtrack)|201933.0   |
|Hold On                                               |198853.0   |
|Days I Will Remember                                  |214240.0   |
|Is There Anybody Out There?                           |229400.0   |
|We Sing. We Dance. We Steal Things.                   |242946.0   |
|We Sing. We Dance. We Steal Things.                   |189613.0   |
|Hunger                                                |205594.0   |
|Episode                                               |244800.0   |
|Love Is a Four Letter Word       

In [17]:
# Leyendo nuevamente el dataset
df = spark.read.csv(
    "/content/dataset.csv",
    header=True,
    multiLine=True,
    quote='"',
    escape='"',
    inferSchema=False
)


In [18]:
# se verifica el esquema
df.printSchema()


root
 |-- _c0: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- track_genre: string (nullable = true)



In [19]:
# convertimos solo las columnas que SI son numéricas
df.select("duration_ms").distinct().show(10, truncate=False)


+-----------+
|duration_ms|
+-----------+
|406103     |
|172857     |
|240200     |
|239026     |
|118388     |
|177826     |
|170426     |
|255826     |
|173933     |
|246506     |
+-----------+
only showing top 10 rows


In [20]:
from pyspark.sql.functions import expr

numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "key", "loudness", "mode", "speechiness",
    "acousticness", "liveness", "valence"
]

for c in numeric_cols:
    df = df.withColumn(c, expr(f"try_cast({c} as double)"))


In [24]:
# se limpian las columnas extra
df = df.dropna(subset=numeric_cols)

In [29]:
from pyspark.sql.functions import expr

numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "key", "loudness", "mode", "speechiness",
    "acousticness", "liveness", "valence"
]

for c in numeric_cols:
    df = df.withColumn(c, expr(f"try_cast({c} as double)"))

df = df.dropna(subset=numeric_cols)


In [30]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="track_genre",
    outputCol="label"
)

df_model = label_indexer.fit(df).transform(df)


In [33]:
from pyspark.sql.functions import expr

cols_to_convert = [
    "instrumentalness",
    "tempo",
    "time_signature"
]

for c in cols_to_convert:
    df_model = df_model.withColumn(c, expr(f"try_cast({c} as double)"))


In [34]:
df_model = df_model.dropna(subset=cols_to_convert)


In [35]:
df_model.printSchema()


root
 |-- _c0: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: double (nullable = true)
 |-- track_genre: string (nullable = true)
 |-- label: double (nullable = false)



In [36]:
df_model = assembler.transform(df_model)


In [37]:
data = df_model.select("features", "label")


In [38]:
data = df_model.select("features", "label")


In [39]:
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())


Train: 79740
Test: 34260


In [40]:
# Entrenar Random Forest

In [41]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

model = rf.fit(train_data)


In [42]:
# Predicciones
predictions = model.transform(test_data)

predictions.select("label", "prediction", "probability").show(10, truncate=False)


+-----+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [45]:
# Se evalua el modelo
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

evaluator.setMetricName("f1")
f1 = evaluator.evaluate(predictions)
print("F1 Score:", f1)


Accuracy: 0.28809106830122594
F1 Score: 0.2535550224747088


In [46]:
# Variables importantes
importances = model.featureImportances

feature_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature"
]

for feature, importance in zip(feature_cols, importances):
    print(feature, importance)


popularity 0.2599969886851565
duration_ms 0.09087202970541941
danceability 0.0965417004246581
energy 0.06499042047933899
key 0.009235717690845744
loudness 0.0636767832008887
mode 0.00922279361041495
speechiness 0.06665493664096188
acousticness 0.10698057804353196
instrumentalness 0.07731044029078958
liveness 0.0363299580681242
valence 0.061953207961087234
tempo 0.049415439784757674
time_signature 0.006819005414025114
